<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void ProductDisplayDelegate();
public delegate void ProductEventHandler(string message);
public delegate bool ProductFilterDelegate(Product product);

public class Product
{
    public string Name { get; set; }
    public decimal Price { get; set; }
    public string Manufacturer { get; set; }
    
    public string ProductCode { get; set; }
    public int StockQuantity { get; set; }
    public DateTime CreatedDate { get; set; }
    public bool IsAvailable { get; set; }
    
    public event ProductEventHandler ProductEvent;

    public Product(string name, decimal price, string manufacturer, string productCode, int stockQuantity)
    {
        Name = name;
        Price = price;
        Manufacturer = manufacturer;
        ProductCode = productCode;
        StockQuantity = stockQuantity;
        CreatedDate = DateTime.Now;
        IsAvailable = stockQuantity > 0;
    }

    protected virtual void OnProductEvent(string message)
    {
        ProductEvent?.Invoke(message);
    }

    public virtual string GetInfo()
    {
        return $"Продукт: {Name}, Цена: {Price:C}, Производитель: {Manufacturer}";
    }

    public virtual void Discount(decimal discountPercentage)
    {
        if (discountPercentage < 0 || discountPercentage > 100)
        {
            OnProductEvent($"Неверный процент скидки: {discountPercentage}%");
            return;
        }
        
        decimal oldPrice = Price;
        Price -= Price * (discountPercentage / 100);
        OnProductEvent($"Применена скидка {discountPercentage}%. Старая цена: {oldPrice:C}, Новая цена: {Price:C}");
    }

    public virtual void Display()
    {
        Console.WriteLine(GetInfo());
        Console.WriteLine($"Код товара: {ProductCode}, В наличии: {StockQuantity}, Доступен: {(IsAvailable ? "Да" : "Нет")}");
    }

    public virtual void UpdateStock(int quantity)
    {
        StockQuantity += quantity;
        IsAvailable = StockQuantity > 0;
        OnProductEvent($"Обновлен запас {Name}. Новое количество: {StockQuantity}");
    }

    public virtual bool ValidateProduct()
    {
        bool isValid = !string.IsNullOrEmpty(Name) && 
                      Price > 0 && 
                      !string.IsNullOrEmpty(Manufacturer) &&
                      !string.IsNullOrEmpty(ProductCode);
        
        if (!isValid)
        {
            OnProductEvent($"Продукт {Name} не прошел валидацию");
        }
        
        return isValid;
    }

    public virtual decimal CalculateShippingCost(decimal baseCost)
    {
        return baseCost + (Price * 0.05m); 
    }
}

public class Electronics : Product
{
    public int WarrantyPeriod { get; set; } 
    public string PowerConsumption { get; set; }
    public string OperatingSystem { get; set; }
    public bool HasBattery { get; set; }

    public Electronics(string name, decimal price, string manufacturer, string productCode, 
                      int stockQuantity, int warrantyPeriod, string powerConsumption, 
                      string operatingSystem, bool hasBattery)
        : base(name, price, manufacturer, productCode, stockQuantity)
    {
        WarrantyPeriod = warrantyPeriod;
        PowerConsumption = powerConsumption;
        OperatingSystem = operatingSystem;
        HasBattery = hasBattery;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Гарантия: {WarrantyPeriod} мес.";
    }

    public override void Discount(decimal discountPercentage)
    {
        if (WarrantyPeriod > 24 && discountPercentage > 20)
        {
            OnProductEvent($"Для товара с гарантией {WarrantyPeriod} мес. максимальная скидка 20%");
            discountPercentage = 20;
        }
        
        base.Discount(discountPercentage);
    }

    public override void Display()
    {
        base.Display();
        Console.WriteLine($"Гарантия: {WarrantyPeriod} мес., Потребление: {PowerConsumption}, ОС: {OperatingSystem}, Аккумулятор: {(HasBattery ? "Да" : "Нет")}");
    }

    public void ExtendWarranty(int additionalMonths)
    {
        WarrantyPeriod += additionalMonths;
        OnProductEvent($"Гарантия продлена на {additionalMonths} мес. Общий срок: {WarrantyPeriod} мес.");
    }

    public decimal CalculateEnergyCost(decimal costPerKwh, int hours)
    {
        if (double.TryParse(PowerConsumption.Replace("W", ""), out double watts))
        {
            return (decimal)(watts * hours / 1000.0) * costPerKwh;
        }
        return 0;
    }
}

public class Clothing : Product
{
    public string Size { get; set; }
    public string Material { get; set; }
    public string Color { get; set; }
    public string Season { get; set; }

    public Clothing(string name, decimal price, string manufacturer, string productCode, 
                   int stockQuantity, string size, string material, string color, string season)
        : base(name, price, manufacturer, productCode, stockQuantity)
    {
        Size = size;
        Material = material;
        Color = color;
        Season = season;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Размер: {Size}";
    }

    public override void Display()
    {
        Console.WriteLine($"Одежда: {Name}");
        Console.WriteLine($"Цена: {Price:C}, Производитель: {Manufacturer}");
        Console.WriteLine($"Размер: {Size}, Материал: {Material}, Цвет: {Color}, Сезон: {Season}");
        Console.WriteLine($"Код товара: {ProductCode}, В наличии: {StockQuantity}");
    }

    public override void Discount(decimal discountPercentage)
    {
        if (Season == "Winter" && DateTime.Now.Month >= 6 && DateTime.Now.Month <= 8)
        {
            discountPercentage += 10; 
            OnProductEvent($"Сезонная скидка +10% для зимней одежды");
        }
        
        base.Discount(discountPercentage);
    }

    public bool TrySize(string customerSize)
    {
        string[] sizes = { "XS", "S", "M", "L", "XL" };
        int currentIndex = Array.IndexOf(sizes, Size);
        int customerIndex = Array.IndexOf(sizes, customerSize);
        
        bool fits = Math.Abs(currentIndex - customerIndex) <= 1;
        OnProductEvent($"Примерка размера {Size} для клиента {customerSize}: {(fits ? "Подходит" : "Не подходит")}");
        
        return fits;
    }

    public string GetCareInstructions()
    {
        return Material switch
        {
            "Cotton" => "Стирка при 40°C, глажка при средней температуре",
            "Wool" => "Ручная стирка, не отжимать",
            "Silk" => "Химчистка или бережная ручная стирка",
            _ => "Следовать инструкциям на ярлыке"
        };
    }
}

public class Books : Product
{
    public string Author { get; set; }
    public string Genre { get; set; }
    public int PageCount { get; set; }
    public string ISBN { get; set; }

    public Books(string name, decimal price, string manufacturer, string productCode, 
                int stockQuantity, string author, string genre, int pageCount, string isbn)
        : base(name, price, manufacturer, productCode, stockQuantity)
    {
        Author = author;
        Genre = genre;
        PageCount = pageCount;
        ISBN = isbn;
    }

    public override string GetInfo()
    {
        return $"Книга: {Name}, Автор: {Author}, Цена: {Price:C}, Жанр: {Genre}";
    }

    public override void Display()
    {
        Console.WriteLine(GetInfo());
        Console.WriteLine($"Страниц: {PageCount}, ISBN: {ISBN}");
        Console.WriteLine($"Код товара: {ProductCode}, В наличии: {StockQuantity}");
    }

    public override void Discount(decimal discountPercentage)
    {
        if (Genre == "Bestseller" && discountPercentage > 15)
        {
            OnProductEvent($"Бестселлеры имеют максимальную скидку 15%");
            discountPercentage = 15;
        }
        
        base.Discount(discountPercentage);
    }

    public string GetReadingTime()
    {
        if (PageCount <= 100) return "1-2 часа";
        if (PageCount <= 300) return "3-6 часов";
        if (PageCount <= 600) return "1-2 дня";
        return "Несколько дней";
    }

    public bool ValidateISBN()
    {
        if (string.IsNullOrEmpty(ISBN) || ISBN.Length != 13) return false;
        
        int sum = 0;
        for (int i = 0; i < 12; i++)
        {
            int digit = ISBN[i] - '0';
            sum += (i % 2 == 0) ? digit : digit * 3;
        }
        
        int checkDigit = (10 - (sum % 10)) % 10;
        return checkDigit == (ISBN[12] - '0');
    }
}


        List<Product> products = new List<Product>
        {
            new Electronics("iPhone 15", 999m, "Apple", "ELEC001", 50, 12, "20W", "iOS", true),
            new Electronics("Samsung TV", 799m, "Samsung", "ELEC002", 30, 36, "150W", "Tizen", false),
            new Clothing("Джинсы", 59.99m, "Levi's", "CLOTH001", 100, "M", "Cotton", "Blue", "All"),
            new Clothing("Зимняя куртка", 199.99m, "North Face", "CLOTH002", 25, "L", "Polyester", "Black", "Winter"),
            new Books("Война и мир", 29.99m, "Эксмо", "BOOK001", 200, "Лев Толстой", "Classic", 1225, "9785171202123"),
            new Books("Преступление и наказание", 24.99m, "АСТ", "BOOK002", 150, "Федор Достоевский", "Bestseller", 671, "9785171489241")
        };

        Console.WriteLine("=== ДЕЛЕГАТЫ ДЛЯ ОТОБРАЖЕНИЯ ===");
        ProductDisplayDelegate displayAll = () =>
        {
            foreach (var product in products)
            {
                product.Display();
                Console.WriteLine();
            }
        };

        displayAll();

        Console.WriteLine("=== ФИЛЬТРАЦИЯ ПРОДУКТОВ ===");
        
        ProductFilterDelegate expensiveFilter = p => p.Price > 100;
        ProductFilterDelegate availableFilter = p => p.IsAvailable;
        ProductFilterDelegate electronicsFilter = p => p is Electronics;

        var expensiveProducts = products.Where(p => expensiveFilter(p));
        Console.WriteLine("\nДорогие товары (> 100$):");
        foreach (var product in expensiveProducts)
        {
            product.Display();
            Console.WriteLine();
        }

        var availableElectronics = products.Where(p => availableFilter(p) && electronicsFilter(p));
        Console.WriteLine("\nДоступная электроника:");
        foreach (var product in availableElectronics)
        {
            product.Display();
            Console.WriteLine();
        }

        Console.WriteLine("=== СОБЫТИЯ ===");
        
        foreach (var product in products)
        {
            product.ProductEvent += (message) => Console.WriteLine($"Событие: {message}");
        }

        Console.WriteLine("=== ДЕМОНСТРАЦИЯ МЕТОДОВ ===");
        
        products[0].Discount(15); // iPhone со скидкой
        products[3].Discount(25); // Зимняя куртка со скидкой
        products[5].Discount(20); // Бестселлер со скидкой

        Console.WriteLine();
        
        products[1].UpdateStock(-5);
        products[2].UpdateStock(50);

        Console.WriteLine();
        
        if (products[0] is Electronics iphone)
        {
            iphone.ExtendWarranty(6);
            decimal energyCost = iphone.CalculateEnergyCost(0.15m, 24);
            Console.WriteLine($"Стоимость энергии за 24 часа: {energyCost:C}");
        }

        if (products[2] is Clothing jeans)
        {
            jeans.TrySize("L");
            Console.WriteLine($"Инструкции по уходу: {jeans.GetCareInstructions()}");
        }

        if (products[4] is Books warAndPeace)
        {
            Console.WriteLine($"Время чтения: {warAndPeace.GetReadingTime()}");
            Console.WriteLine($"ISBN валиден: {warAndPeace.ValidateISBN()}");
        }

        Console.WriteLine("\n=== ГРУППИРОВКА ПРОДУКТОВ ===");
        
        var groupedByType = products.GroupBy(p => p.GetType().Name);
        foreach (var group in groupedByType)
        {
            Console.WriteLine($"\n{group.Key}:");
            foreach (var product in group)
            {
                Console.WriteLine($"  - {product.Name}");
            }
        }

        Console.WriteLine("\n=== СОРТИРОВКА ПО ЦЕНЕ ===");
        var sortedByPrice = products.OrderBy(p => p.Price);
        foreach (var product in sortedByPrice)
        {
            Console.WriteLine($"{product.Name}: {product.Price:C}");
        }
    

=== ДЕЛЕГАТЫ ДЛЯ ОТОБРАЖЕНИЯ ===
Продукт: iPhone 15, Цена: ¤999.00, Производитель: Apple, Гарантия: 12 мес.
Код товара: ELEC001, В наличии: 50, Доступен: Да
Гарантия: 12 мес., Потребление: 20W, ОС: iOS, Аккумулятор: Да

Продукт: Samsung TV, Цена: ¤799.00, Производитель: Samsung, Гарантия: 36 мес.
Код товара: ELEC002, В наличии: 30, Доступен: Да
Гарантия: 36 мес., Потребление: 150W, ОС: Tizen, Аккумулятор: Нет

Одежда: Джинсы
Цена: ¤59.99, Производитель: Levi's
Размер: M, Материал: Cotton, Цвет: Blue, Сезон: All
Код товара: CLOTH001, В наличии: 100

Одежда: Зимняя куртка
Цена: ¤199.99, Производитель: North Face
Размер: L, Материал: Polyester, Цвет: Black, Сезон: Winter
Код товара: CLOTH002, В наличии: 25

Книга: Война и мир, Автор: Лев Толстой, Цена: ¤29.99, Жанр: Classic
Страниц: 1225, ISBN: 9785171202123
Код товара: BOOK001, В наличии: 200

Книга: Преступление и наказание, Автор: Федор Достоевский, Цена: ¤24.99, Жанр: Bestseller
Страниц: 671, ISBN: 9785171489241
Код товара: BOOK002, В